Работу выполнил: Резников Иван, P3122

# Инструменты для работы с языком

... или зачем нужна предобработка.

Раньше мы смотрели на светлую сторону анализа данных - построение моделей. Теперь попробуем глубже посмотреть на часть про предобработку данных. Задача предобработки особенно актуальна, если мы имеем дело с текстами.

## Задача: классификация твитов по тональности

У нас есть выборка из твитов.
Нам известна эмоциональная окраска каждого твита из выборки: положительная или отрицательная. Задача состоит в построении модели, которая по тексту твита предсказывает его эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем выборку ([источник](http://study.mokoron.com/)): [положительные](https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv), [отрицательные]( https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv).

In [1]:
!wget https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv
!wget https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv

--2026-05-21 15:52:48--  https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26233379 (25M) [application/octet-stream]
Saving to: ‘positive.csv’

positive.csv        100%[===================>]  25.02M  62.4MB/s    in 0.4s    

2026-05-21 15:52:50 (62.4 MB/s) - ‘positive.csv’ saved [26233379/26233379]

--2026-05-21 15:52:50--  https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24450101 (23M)

In [2]:
import pandas as pd # библиотека для удобной работы с датафреймами
import numpy as np # библиотека для удобной работы со списками и матрицами

# библиотека, где реализованы основные алгоритмы машинного обучения
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [3]:
!head positive.csv

"408906692374446080";"1386325927";"pleease_shut_up";"@first_timee хоть я и школота, но поверь, у нас то же самое :D общество профилирующий предмет типа)";"1";"0";"0";"0";"7569";"62";"61";"0"
"408906692693221377";"1386325927";"alinakirpicheva";"Да, все-таки он немного похож на него. Но мой мальчик все равно лучше:D";"1";"0";"0";"0";"11825";"59";"31";"2"
"408906695083954177";"1386325927";"EvgeshaRe";"RT @KatiaCheh: Ну ты идиотка) я испугалась за тебя!!!";"1";"0";"1";"0";"1273";"26";"27";"0"
"408906695356973056";"1386325927";"ikonnikova_21";"RT @digger2912: ""Кто то в углу сидит и погибает от голода, а мы ещё 2 порции взяли, хотя уже и так жрать не хотим"" :DD http://t.co/GqG6iuE2…";"1";"0";"1";"0";"1549";"19";"17";"0"
"408906761416867842";"1386325943";"JumpyAlex";"@irina_dyshkant Вот что значит страшилка :D
Но блин,посмотрев все части,у тебя создастся ощущение,что авторы курили что-то :D";"1";"0";"0";"0";"597";"16";"23";"1"
"408906761769598976";"1386325943";"JustinB94262583";"ну любишь и

In [4]:
!tail negative.csv

"425137932283158528";"1390195756";"pazyfevesity";"RT @qelasocadij: Скажите пожалуйста, как у человека может быть 1000 одноклассников? O_o";"-1";"0";"1";"0";"2168";"171";"132";"0"
"425137934443233281";"1390195756";"Sonya_Star_14";"У нас физ ра на улице
Пака линт:(
Через 45 минут приду пхжааххв";"-1";"0";"0";"0";"12722";"638";"567";"2"
"425138035089358848";"1390195780";"evalesana";"Нас сегодня отказались принять в сад, типа мы плачем(( #королев пойду ругаться сейчас, по крайне мере выяснять, что за фигня";"-1";"0";"0";"0";"4101";"166";"151";"6"
"425138243257253888";"1390195830";"Yanch_96";"Но не каждый хочет что то исправлять:( http://t.co/QNODDQzuZ7";"-1";"0";"0";"0";"1138";"32";"46";"0"
"425138339503943682";"1390195853";"tkit_on";"скучаю так :-( только @taaannyaaa вправляет мозги, но я все равно скучаю";"-1";"0";"0";"0";"4822";"38";"32";"0"
"425138437684215808";"1390195876";"ckooker1";"Вот и в школу, в говно это идти уже надо(";"-1";"0";"0";"1";"165";"13";"16";"0"
"425138490452344832";

Откроем файлы и создадим массив из текстов и правильных меток для твитов.
Сначала идут положительные твиты, потом отрицательные.

In [5]:
# загружаем положительные твиты
# TODO #2
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = ['positive'] * len(positive)
# загружаем отрицательные твиты
# TODO #3
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = ['negative'] * len(negative)
# соединяем вместе
# TODO #4
df = pd.concat([positive, negative])


In [6]:
df

,text,label
0,"@first_timee хоть я и школота, но поверь, у на...",positive
1,"Да, все-таки он немного похож на него. Но мой ...",positive
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",positive
4,@irina_dyshkant Вот что значит страшилка :D\nН...,positive
...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,negative
111919,скучаю так :-( только @taaannyaaa вправляет мо...,negative
111920,"Вот и в школу, в говно это идти уже надо(",negative
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",negative


Посмотрим на полученные данные:

In [7]:
df.sample(5, random_state=40)

,text,label
15931,RT @Blawar_1337: Теперь у нас с @Wake_UA появи...,positive
59532,с днём рождения зайка*))) ухх погуляем мы сего...,positive
47185,RT @Shumkova0406199: @ann_safina Вов вов вов А...,negative
42002,"Надо выдернуть звуковую дорожку из ""Доктора Ка...",positive
109035,@_hassliebe_ может все таки на этой неделе вер...,negative


Разбиваем данные на обучающую и тестовую выборки с помощью функции ```train_test_split()``` из **sklearn**:


In [8]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label)


print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(170125,) (56709,) (170125,) (56709,)


In [9]:
y_train[:10]

,label
8109,negative
65570,negative
72573,positive
101541,negative
45941,negative
56766,negative
7242,negative
27600,negative
78134,negative
54459,negative


In [10]:
y_train.value_counts()

,count
label,
positive,86028
negative,84097


## Baseline: классификация необработанных n-грамм

* Сейчас мы попробуем получить преобразование предложений в численный вектор, с которым может работать стандартный алгоритм машинного обучения, такой как логистическая регрессия.
* Для этого нам понадобится познакомиться с понятием n-gram - самых мелких элементов предложения, с которыми можно работать.
* Подсчитав количество этих n-грам в предложениях, мы получим искомые численные представления.

## Что такое n-граммы:

Самые мелкие структуры языка, с которыми мы работаем, называются **n-граммами**.
У n-граммы есть параметр n - количество слов, которые попадают в такое представление текста.
* Если n = 1 - то мы смотрим на то, сколько раз каждое слово встретилось в тексте. Получаем _униграммы_
* Если n = 2 - то мы смотрим на то, сколько раз каждая пара подряд идущих слов, встретилась в тексте. Получаем _биграммы_

**Функция** для работы с n-граммами реализована в библиотке **nltk** (Natural Language ToolKit), импортируем эту функцию:

In [11]:
from nltk import ngrams

Прежде чем получать n-граммы, нужно разделить предложение на отдельные слова.  Для этого используем метод ```split()```.

In [12]:
sentence = 'Если б мне платили каждый раз'.split()
sentence

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Чтобы получить n-грамму для такой последовательности, используем функцию ```ngrams()```.

На вход передается два параметра:
* лист с разделенным на отдельные слова предложением (у нас он хранится в переменной ```sent```);
* параметр n, определяющий, какой тип n-грамм мы хотим получить.


Чтобы полученный объект отобразить, делаем из него ```list```.

In [13]:
list(ngrams(sentence, 1)) # униграммы

[('Если',), ('б',), ('мне',), ('платили',), ('каждый',), ('раз',)]

Аналогично мы можем получить биграммы - для этого заменяем параметр **n** в функции **ngrams** с 1 на 2.

In [14]:
list(ngrams(sentence, 2)) # биграммы

[('Если', 'б'),
 ('б', 'мне'),
 ('мне', 'платили'),
 ('платили', 'каждый'),
 ('каждый', 'раз')]

In [15]:
list(ngrams(sentence, 3)) # триграммы

[('Если', 'б', 'мне'),
 ('б', 'мне', 'платили'),
 ('мне', 'платили', 'каждый'),
 ('платили', 'каждый', 'раз')]

In [16]:
list(ngrams(sentence, 5)) # ... пентаграммы?

[('Если', 'б', 'мне', 'платили', 'каждый'),
 ('б', 'мне', 'платили', 'каждый', 'раз')]

### Векторизаторы

Векторизатор преобразует слово или набор слов в числовой вектор, понятный алгоритму машинного обучения, который привык работать с числовыми табличными данными.

Ниже - пример преобразования слов в двумерных вектор, каждому слову соответствует точка на плоскости.

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На начальном этапе нам будет достаточно тех инструментов, которые уже есть в знакомой нам библиотеке **sklearn**.

In [17]:
from sklearn.linear_model import LogisticRegression # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer # модель "мешка слов", см. далее

Самый простой способ извлечь признаки из текстовых данных -- векторизаторы: `CountVectorizer` и `TfidfVectorizer`

Объект `CountVectorizer` делает следующую вещь:
* строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` -- количество слов или n-грам во всём корпусе
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1jHmkrGZTMawM46Yzxh243Ur1y5pYKzrl"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На рисунке пример векторизации для униграмм, но можно использовать любые n-граммы. Для этого у объекта ```CountVectorizer()``` есть параметр **ngram_range**, который отвечает за то, какие n-граммы мы используем в качестве признаов:<br/>
ngram_range=(1, 1) -- униграммы<br/>
ngram_range=(3, 3) -- триграммы<br/>
ngram_range=(1, 3) -- униграммы, биграммы и триграммы.

<a href="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D" target="_blank"><img src="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D"
alt="IMAGE ALT TEXT HERE" width="800" border="0" /></a>

Инициализируем ```CountVectorizer()```, указав в качестве признаков униграммы:

In [18]:
vectorizer = CountVectorizer(ngram_range=(1,1))

После инициализации _vectorizer_ можно обучить на наших данных.

Для обучения используем обучающую выборку ```x_train```, но в отличие от классификатора мы используем метод ```fit_transform()```: сначала обучаем наш векторизатор, а потом сразу применяем его к нашему набору данных. Это похоже на то, как мы работали с label encoderом и one-hot-encoderом.


In [19]:
# TODO #8
vectorized_x_train = vectorizer.fit_transform(x_train)

Так как результат не зависит от порядка слов в текстах, то говорят, что такая модель представления текстов в виде векторов получается из *гипотезы представления текста как мешка слов*

В vectorizer.vocabulary_ лежит словарь, отображение слов в их индексы:

In [20]:
list(vectorizer.vocabulary_.items())[:10]

[('вот', 115072),
 ('новая', 168689),
 ('диета', 125879),
 ('называется', 162866),
 ('она', 173585),
 ('доме', 127596),
 ('сгорела', 207715),
 ('плита', 183427),
 ('rt', 74751),
 ('elenium72', 28897)]

В нашей выборке 170125 текстов (твитов), в них встречается 243760 разных слов.

In [21]:
vectorized_x_train.shape

(170125, 243852)

Так как теперь у нас есть **численное представление** и набор входных признаков, то мы можем обучить модель логистической регрессии (или любую другую из тех, на которые мы смотрели раньше, например, случайный лес)

In [22]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

С тестовыми данными нужно проделать то же самое, что и с данными для обучения: сделать из текстов вектора, которые можно передавать в классификатор для прогноза класса объекта.

У нас уже есть обученный векторизатор ```vectorizer```, поэтому используем метод ```transform()``` (просто применить его), а не ```fit_transform``` (обучить и применить).

In [23]:
vectorized_x_test = vectorizer.transform(x_test)

Как раньше, для получения прогноза у обученного классификатора используем метод ```predict()```.

С помощью функции ```classification_report()```, которая считает сразу несколько метрик качества классификации, посмотрим на то, насколько хорошо мы предсказываем положительную или отрицательную тональность твита .

In [24]:
pred_res = clf.predict(vectorized_x_test)

print(classification_report(y_test, pred_res))

              precision    recall  f1-score   support

    negative       0.76      0.77      0.76     27826
    positive       0.77      0.77      0.77     28883

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



## Бонус*: триграммы

Попробуем сделать то же самое, используя в качестве признаков триграммы:

In [25]:
# TODO #12

# инициализируем векторайзер
vectorizer_3 = CountVectorizer(ngram_range=(3,3))
# обучаем его и сразу применяем к x_train
vectorized_x_train_3 = vectorizer_3.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train_3, y_train)
# применяем обученный векторизатор к тестовым данным
vectorized_x_test_3 = vectorizer_3.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred = clf.predict(vectorized_x_test_3)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.71      0.47      0.57     27826
    positive       0.62      0.82      0.70     28883

    accuracy                           0.65     56709
   macro avg       0.67      0.64      0.63     56709
weighted avg       0.66      0.65      0.64     56709



Как вы думаете, почему в результатах теперь такой разброс по сравнению с униграммами?

## Бонус**: TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений выдает **tf-idf** каждого слова.

Как считается tf-idf:

**TF (term frequency)** – относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_{t}}{\sum_k n_{k}} $$

**IDF (inverse document frequency)** – обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \mbox{log} \frac{|D|}{|{d : t \in d}|} $$

Перемножаем их:
$$TFIDF(t, d, D) = TF(t,d) \times IDF(i, D)$$

Сакральный смысл: если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом
количестве документов, у него высокий TF-IDF.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

Действуем аналогично, как с ```CountVectorizer()```:

In [27]:
# TODO #13

# инициализируем векторизатор, в качестве переменных используем униграммы
tfidfvectorizer = TfidfVectorizer(ngram_range=(1,1))
# обучаем его и сразу применяем к x_train
tfidf_vectorized_x_train = tfidfvectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
# применяем обученный векторизатор к тестовым данным
clf.fit(tfidf_vectorized_x_train, y_train)

# получаем предсказания и выводим информацию о качестве
tfidf_vectorized_x_test  = tfidfvectorizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_vectorized_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.77      0.73      0.75     27826
    positive       0.75      0.79      0.77     28883

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



In [28]:
# TODO #14

# инициализируем векторизатор, в качестве переменных используем пентаграммы
tfidfvectorizer = TfidfVectorizer(ngram_range=(5, 5))
# обучаем его и сразу применяем к x_train
tfidf_vectorized_x_train = tfidfvectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
# применяем обученный векторизатор к тестовым данным
clf.fit(tfidf_vectorized_x_train, y_train)

# получаем предсказания и выводим информацию о качестве
tfidf_vectorized_x_test  = tfidfvectorizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_vectorized_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.95      0.11      0.20     27826
    positive       0.54      0.99      0.70     28883

    accuracy                           0.56     56709
   macro avg       0.74      0.55      0.45     56709
weighted avg       0.74      0.56      0.45     56709



## Токенизация

Токенизировать - значит, поделить текст на части: слова, ключевые слова, фразы, символы и т.д., иными словами **токены**.

Самый наивный способ токенизировать текст - разделить с помощью функции `split()`. Но `split` упускает очень много всего, например, не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем, поэтому лучше использовать готовые токенизаторы.

In [29]:
import nltk # уже знакомая нам библиотека nltk
from nltk.tokenize import word_tokenize # готовый токенизатор библиотеки nltk

Чтобы использовать токенизатор ```word_tokenize```, нужно сначала скачать данные для nltk о пунктуации и стоп-словах. Это просто требование nltk, поэтому, особо не задумываясь, запустите следующую ячейку:  

In [30]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Применим токенизацию:

In [31]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

Если использовать просто ```split()```, то грустный смайлик :( не отделяется от слова "исправлять":

In [32]:
example.split()

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять:(']

В nltk вообще есть довольно много токенизаторов:

In [33]:
from nltk import tokenize
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы в строке для начала и конца каждого слова-токена:

In [34]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

Некторые токенизаторы ведут себя специфично:

In [35]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

А некоторые -- вообще не для текста на естественном языке:

In [36]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

**Правильный токенизатор подбирается исходя из требований задачи!**

## Стоп-слова и пунктуация

**Стоп-слова** - это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конретном документе. Для модели это просто шум. А шум нужно убирать. По аналогичной причине убирают и пунктуацию.

In [37]:
# импортируем стоп-слова из библиотеки nltk
from nltk.corpus import stopwords

# посмотрим на стоп-слова для русского языка
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

*Знаки* пунктуации лучше импортировать из модуля **String**. В нем хранятся различные наборы констант для работы со строками (пунктуация, алфавит и др.).

In [38]:
from string import punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

Объединим стоп-слова и знаки пунктуации вместе и запишем в переменную ```noise```:

In [39]:
noise = stopwords.words('russian') + list(punctuation)

Теперь нужно обучать нашу модель с учетом новых знаний про токенизацию и стоп-слова.

Для этого мы можем собрать новый векторизатор, передав ему на вход:
* какие n-граммы нам нужны, параметр **ngram_range**;
* какой токенизатор мы используем, параметр **tokenizer**;
* какие у нас стоп-слова, параметр **stop_words**.

*Напоминание:* мы используем готовый токенизатор ```word_tokenize```, а стоп-слова хранятся в переменной ```noise```

In [40]:
# инициализируем умный векторайзер
# TODO #16 с word_tokenize
smart_tokenizer = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize,
                                  stop_words=noise)

In [41]:
# TODO #17

# обучаем его и сразу применяем к x_train
smart_x_train = smart_tokenizer.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)

# применяем обученный векторайзер к тестовым данным
clf.fit(smart_x_train, y_train)

# получаем предсказания и выводим информацию о качестве
smart_x_test = smart_tokenizer.transform(x_test)

pred = clf.predict(smart_x_test)
print(classification_report(y_test, pred))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

    negative       0.76      0.80      0.78     27826
    positive       0.80      0.76      0.78     28883

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709



Получилось лучше: accuracy выше, а также заметно подрос recall у негативного класса.

Что ещё можно сделать?

## Бонус*: Лемматизация

**Лемматизация** – это сведение разных форм одного слова к начальной форме – **лемме**. Почему это хорошо?
* Во-первых, естественно рассматривать как отдельный признак каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

Для русского есть хороших лемматизатор pymorphy.

### [Pymorphy](http://pymorphy2.readthedocs.io/en/latest/)
Это модуль на питоне, довольно быстрый и с кучей функций.

In [42]:
# устанавливаем pymorphy3
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 838.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 25.5 MB/s eta 0:00:00


В pymorphy2 для морфологического анализа слов есть ```MorphAnalyzer()```:

In [43]:
from pymorphy3 import MorphAnalyzer
# создадим объект pymorphy3_analyzer импортированного класса

# TODO #18
pymorphy3_analyzer = MorphAnalyzer()

pymorphy3 работает с отдельными словами. Если дать ему на вход предложение - он его просто не лемматизирует, т.к. не понимает:

In [44]:
sent = ['Если', 'б', 'мне', 'платили', 'каждый', 'раз']
sent

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Лемматизируем слово "платили" из предложения ```sent``` с помощью метода ```parse()```:

In [45]:
ana = pymorphy3_analyzer.parse(sent[3])
ana

[Parse(word='платили', tag=OpencorporaTag('VERB,impf,tran plur,past,indc'), normal_form='платить', score=1.0, methods_stack=((DictionaryAnalyzer(), 'платили', 2471, 10),))]

Выведем его нормальную форму:

In [46]:
ana[0].normal_form

'платить'

## О важности эксплоративного анализа

Но иногда пунктуация бывает и не шумом - главное отталкиваться от задачи. Что будет если вообще не убирать пунктуацию?

In [47]:
# TODO #19

# инициализируем умный векторайзер stop-words не используем
smart_tokenizer = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize)

# обучаем его и сразу применяем к x_train
smart_x_train = smart_tokenizer.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)

# применяем обученный векторайзер к тестовым данным
clf.fit(smart_x_train, y_train)

# получаем предсказания и выводим информацию о качестве
smart_x_test = smart_tokenizer.transform(x_test)

pred = clf.predict(smart_x_test)
print(classification_report(y_test, pred))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     27826
    positive       1.00      1.00      1.00     28883

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Шок! Стоило оставить пунктуацию -- и все метрики равны 1. Как это получилось? Среди неё были очень значимые токены (как вы думаете, какие?).

Посмотрим, как один из супер-значительных токенов справится с классификацией безо всякого машинного обучения:

In [48]:
# TODO #20

def my_clf(texts, token=" "):
    return np.array(['positive' if token in text else 'negative' for text in texts])

pred = my_clf(x_test, token=")")

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.85      1.00      0.92     27826
    positive       1.00      0.83      0.91     28883

    accuracy                           0.91     56709
   macro avg       0.92      0.91      0.91     56709
weighted avg       0.93      0.91      0.91     56709



## Символьные n-граммы

Теперь в качестве фичей используем, например, униграммы символов. Для этого необходимо установить в ```CountVectorizer()``` параметр ```analyzer = 'char'```, то есть анализировать символы.

In [49]:
# TODO #21

# инициализируем векторайзер для символов
symb_vectorizer = CountVectorizer(analyzer='char')
# обучаем его и сразу применяем к x_train
symb_x_train = symb_vectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
# применяем обученный векторайзер к тестовым данным
clf.fit(symb_x_train, y_train)
# получаем предсказания и выводим информацию о качестве
symb_x_test = symb_vectorizer.transform(x_test)

pred = clf.predict(symb_x_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       1.00      0.99      1.00     27826
    positive       0.99      1.00      1.00     28883

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Из предыдущего раздела уже понятно, почему на этих данных точность равна 1.

Символьные n-граммы используются, например, для задачи определения языка. Ещё одна замечательная особенность признаков-символов - для них не нужна токенизация и лемматизация, можно использовать такой подход для языков, у которых нет готовых анализаторов.

# Самостоятельная работа

1. Изучите материал, представленный в борде.
2. Выполните все ячейки и получите результаты.
3. Приведите результаты таблицы classification_report в под этим заданием для модели LogisticRegression
4. Примените 2 альтернативных использованному алгоритму для решения задачи классификации (для примера XGBClassifier и еще какой-то один) и получите результаты в таблице classification_report
5. Для XGBClassifier вам потребуется задать параметры
```learning_rate=0.1, n_estimators=1000, max_depth=5, min_child_weight=3, gamma=0.2, subsample=0.6, colsample_bytree=1.0, objective='binary:logistic', nthread=4, scale_pos_weight=1, seed=27```

6. В разделе TF-IDF векторизация по аналогии с униграммами и пентаграммами вычислите classification_report для биграмм, триграмм опубликуйте результаты в отчете и укажите изменилась ли точность f1-score при их использовании по сравнению с униграммами и пентаграммами.

In [50]:
import pandas as pd
from IPython.display import display

all_results = []

def save_result(name, y_true, y_pred, target_names=None):
    """Сохраняет метрики модели и печатает classification_report."""
    report = classification_report(y_true, y_pred,
                                   target_names=target_names,
                                   output_dict=True)
    class_keys = [k for k in report.keys()
                  if k not in ('accuracy', 'macro avg', 'weighted avg')]
    neg_key = class_keys[0]
    pos_key = class_keys[1] if len(class_keys) > 1 else class_keys[0]

    all_results.append({
        'Метод': name,
        'Accuracy': round(report['accuracy'], 4),
        'F1 (negative)': round(report[neg_key]['f1-score'], 4),
        'F1 (positive)': round(report[pos_key]['f1-score'], 4),
        'F1 (macro avg)': round(report['macro avg']['f1-score'], 4),
    })
    print(f'=== {name} ===')
    print(classification_report(y_true, y_pred, target_names=target_names))

---

Бейзлайн:




In [51]:
save_result('LogisticRegression — Baseline (UniGrams)', y_test, pred_res)

=== LogisticRegression — Baseline (UniGrams) ===
              precision    recall  f1-score   support

    negative       0.76      0.77      0.76     27826
    positive       0.77      0.77      0.77     28883

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



XGBoost

In [52]:
!pip install -q xgboost

In [53]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# числовые метки классов для XGBoost
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

xgb = XGBClassifier(
    learning_rate=0.1,
    n_estimators=1000,
    max_depth=5,
    min_child_weight=3,
    gamma=0.2,
    subsample=0.6,
    colsample_bytree=1.0,
    objective='binary:logistic',
    nthread=4,
    scale_pos_weight=1,
    seed=27
)

xgb.fit(vectorized_x_train, y_train_enc)
xgb_pred = xgb.predict(vectorized_x_test)
save_result('XGBClassifier (CountVectorizer [Unigrams])',
            y_test_enc, xgb_pred,
            target_names=le.classes_)

=== XGBClassifier (CountVectorizer [Unigrams]) ===
              precision    recall  f1-score   support

    negative       0.74      0.67      0.71     27826
    positive       0.71      0.77      0.74     28883

    accuracy                           0.72     56709
   macro avg       0.73      0.72      0.72     56709
weighted avg       0.73      0.72      0.72     56709



RandomForest

In [55]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)
rf.fit(vectorized_x_train, y_train)
rf_pred = rf.predict(vectorized_x_test)

save_result('RandomForest (CountVectorizer [Unigrams])', y_test, rf_pred)

=== RandomForest (CountVectorizer [Unigrams]) ===
              precision    recall  f1-score   support

    negative       0.79      0.31      0.45     27826
    positive       0.58      0.92      0.71     28883

    accuracy                           0.62     56709
   macro avg       0.68      0.62      0.58     56709
weighted avg       0.68      0.62      0.58     56709



TF-IDF

In [56]:
tfidf_2 = TfidfVectorizer(ngram_range=(2, 2))
# обучаем его и сразу применяем к x_train
tfidf_x_train_2 = tfidf_2.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf_2 = LogisticRegression(random_state=42, max_iter=1000)
clf_2.fit(tfidf_x_train_2, y_train)
# применяем обученный векторизатор к тестовым данным
tfidf_x_test_2 = tfidf_2.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred_2 = clf_2.predict(tfidf_x_test_2)

save_result('LogisticRegression (TF-IDF [BiGrams])', y_test, pred_2)

=== LogisticRegression (TF-IDF [BiGrams]) ===
              precision    recall  f1-score   support

    negative       0.72      0.66      0.69     27826
    positive       0.70      0.75      0.72     28883

    accuracy                           0.71     56709
   macro avg       0.71      0.71      0.71     56709
weighted avg       0.71      0.71      0.71     56709



Триграммы:

In [57]:
tfidf_3 = TfidfVectorizer(ngram_range=(3, 3))
# обучаем его и сразу применяем к x_train
tfidf_x_train_3 = tfidf_3.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf_3 = LogisticRegression(random_state=42, max_iter=1000)
clf_3.fit(tfidf_x_train_3, y_train)
# применяем обученный векторизатор к тестовым данным
tfidf_x_test_3 = tfidf_3.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred_3 = clf_3.predict(tfidf_x_test_3)

save_result('LogisticRegression (TF-IDF [TriGrams])', y_test, pred_3)


=== LogisticRegression (TF-IDF [TriGrams]) ===
              precision    recall  f1-score   support

    negative       0.72      0.46      0.56     27826
    positive       0.61      0.83      0.71     28883

    accuracy                           0.65     56709
   macro avg       0.67      0.64      0.63     56709
weighted avg       0.67      0.65      0.63     56709



In [58]:
df_results = pd.DataFrame(all_results).set_index('Метод')

display(
    df_results.style
    .format('{:.4f}')
    .background_gradient(cmap='Blues',
                         subset=['Accuracy', 'F1 (negative)',
                                 'F1 (positive)', 'F1 (macro avg)'])
    .set_caption('Сравнение методов классификации твитов по тональности')
)


,Accuracy,F1 (negative),F1 (positive),F1 (macro avg)
Метод,,,,
LogisticRegression — Baseline (UniGrams),0.7674,0.7642,0.7704,0.7673
XGBClassifier (CountVectorizer [Unigrams]),0.7242,0.7057,0.7405,0.7231
RandomForest (CountVectorizer [Unigrams]),0.6208,0.4471,0.7115,0.5793
LogisticRegression (TF-IDF [BiGrams]),0.7084,0.6906,0.7243,0.7075
LogisticRegression (TF-IDF [TriGrams]),0.6471,0.5597,0.7056,0.6327


**Вывод:**

Сравним использованные подходы:
- **Бейзлайн** (LogisticRegression + CountVectorizer, уни-граммы) показал accuracy ~0.77 и f1-score ~0.77 — хороший отправной результат.
- **XGBClassifier** с заданными гиперпараметрами позволяет оценить влияние бустинга на качество классификации.
- **RandomForestClassifier** — ещё один ансамблевый метод, результаты которого можно сравнить с LogisticRegression.
- В разделе TF-IDF: с биграммами качество изменяется по сравнению с уни-граммами, а с триграммами f1-score значительно падает (так же, как в разделе с CountVectorizer + триграммами) - совпадений из трёх слов подряд в тестовой выборке намного меньше, что ведёт к разреженности. Уни-граммы в данной задаче обеспечивают наилучший баланс между реколлом и пресиженом